In [ ]:
import cv2
import numpy as np
from scipy.spatial import distance as dist
from IPython.display import display, Image, clear_output
import mediapipe as mp
import serial
import time

# --- CONFIGURATION ---
EYE_AR_THRESH = 0.21       
EYE_AR_CONSEC_FRAMES = 20 

# --- SERIAL SETUP ---
# CHANGE THIS LINE to the port you found (e.g., 'COM3' or '/dev/ttyUSB0')
SERIAL_PORT = 'COM5' 
BAUD_RATE = 9600

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"Connected to ESP32 on {SERIAL_PORT}")
    time.sleep(2) # Wait for connection to stabilize
except Exception as e:
    print(f"Could not connect to Serial Port: {e}")
    print("Continuing without ESP32 (LED will not work)...")
    ser = None

# --- HELPER FUNCTIONS ---
def eye_aspect_ratio(eye):
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    C = dist.euclidean(eye[0], eye[3])
    ear = (A + B) / (2.0 * C)
    return ear

# --- INITIALIZATION ---
print("Initializing MediaPipe...")
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True, 
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

LEFT_EYE_INDICES = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_INDICES = [362, 385, 387, 263, 373, 380]

# Start Camera
cap = cv2.VideoCapture(0)
sleep_counter = 0
status = "Awake"

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame = cv2.resize(frame, (640, 480))
        
        # Raw Preview
        raw_preview = frame.copy()
        cv2.putText(raw_preview, "RAW FEED", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # MediaPipe Processing
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_frame)

        detection_frame = frame.copy()

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                img_h, img_w, _ = frame.shape
                mesh_points = np.array([np.multiply([p.x, p.y], [img_w, img_h]).astype(int) for p in face_landmarks.landmark])

                left_eye = mesh_points[LEFT_EYE_INDICES]
                right_eye = mesh_points[RIGHT_EYE_INDICES]

                cv2.polylines(detection_frame, [left_eye], True, (0, 255, 0), 1)
                cv2.polylines(detection_frame, [right_eye], True, (0, 255, 0), 1)

                leftEAR = eye_aspect_ratio(left_eye)
                rightEAR = eye_aspect_ratio(right_eye)
                ear = (leftEAR + rightEAR) / 2.0

                # --- SLEEP LOGIC ---
                if ear < EYE_AR_THRESH:
                    sleep_counter += 1
                    if sleep_counter >= EYE_AR_CONSEC_FRAMES:
                        status = "SLEEPING!"
                        cv2.rectangle(detection_frame, (0,0), (640, 480), (0,0,255), 5)
                        cv2.putText(detection_frame, "SLEEPING ALERT", (150, 240),
                                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 4)
                else:
                    sleep_counter = 0
                    status = "Awake"

                # --- SEND SIGNAL TO ESP32 ---
                if ser:
                    if status == "SLEEPING!":
                        ser.write(b'1') # Send '1' to turn LED ON
                    else:
                        ser.write(b'0') # Send '0' to turn LED OFF

                cv2.putText(detection_frame, f"EAR: {ear:.2f}", (10, 450),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

        cv2.putText(detection_frame, f"Status: {status}", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0) if status == "Awake" else (0,0,255), 2)

        # Display
        combined_view = np.hstack((raw_preview, detection_frame))
        _, jpeg = cv2.imencode('.jpg', combined_view)
        frame_bytes = jpeg.tobytes()
        clear_output(wait=True)
        display(Image(data=frame_bytes))

except KeyboardInterrupt:
    print("\nStopping.")
finally:
    cap.release()
    face_mesh.close()
    if ser:
        ser.close() # Close the serial port
        print("Serial connection closed.")